# Практична робота №20
## Лінійне програмування: постановка задачі, розв'язання засобами scipy
**Виконав:** Хрустовський Павло  
**Група:** ІТ-42  
**Варіант:** 6 (Чернігів)  
**№ у журналі:** 26


In [1]:
from scipy.optimize import linprog

# Максимізація прибутку F = 55*x1 + 40*x2 -> min(-55*x1 - 40*x2)
c_obj = [-55, -40]

# Матриця технологічних обмежень ресурсів:
# 2*x1 + 3*x2 <= 210
# 6*x1 + 2*x2 <= 250
# 2*x1 + 1*x2 <= 125
A_constraints = [
    [2, 3],
    [6, 2],
    [2, 1]
]

b_limits = [210, 250, 125]
bounds_vars = [(0, None), (0, None)]

opt_res = linprog(c_obj, A_ub=A_constraints, b_ub=b_limits, bounds=bounds_vars, method='highs')

print(f"Оптимальний обсяг випуску продукції: x1 = {opt_res.x[0]:.2f}, x2 = {opt_res.x[1]:.2f}")
print(f"Максимальний сумарний прибуток: {-opt_res.fun:.2f} грн")
print(f"Статус розв'язання: {opt_res.status} ({opt_res.message})")


Оптимальний обсяг випуску продукції: x1 = 23.57, x2 = 54.29
Максимальний сумарний прибуток: 3467.86 грн
Статус розв'язання: 0 (Optimization terminated successfully. (HiGHS Status 7: Optimal))


In [2]:
print("Аналіз використання технологічних ресурсів:")
used_amounts = [
    2 * opt_res.x[0] + 3 * opt_res.x[1],
    6 * opt_res.x[0] + 2 * opt_res.x[1],
    2 * opt_res.x[0] + 1 * opt_res.x[1]
]

for idx, (used, limit) in enumerate(zip(used_amounts, b_limits), 1):
    slack_val = limit - used
    is_active = abs(slack_val) < 1e-5
    status_str = "АКТИВНЕ (дефіцитний ресурс, запас = 0)" if is_active else f"НЕАКТИВНЕ (лишок = {slack_val:.2f})"
    print(f"Ресурс {idx}: витрачено {used:.2f} із {limit}. Обмеження: {status_str}")


Аналіз використання технологічних ресурсів:
Ресурс 1: витрачено 210.00 із 210. Обмеження: АКТИВНЕ (дефіцитний ресурс, запас = 0)
Ресурс 2: витрачено 250.00 із 250. Обмеження: АКТИВНЕ (дефіцитний ресурс, запас = 0)
Ресурс 3: витрачено 101.43 із 125. Обмеження: НЕАКТИВНЕ (лишок = 23.57)


In [3]:
# Збільшення запасу активного ресурсу 1 на 15%
limits_sens_active = b_limits.copy()
limits_sens_active[0] *= 1.15

res_active_mod = linprog(c_obj, A_ub=A_constraints, b_ub=limits_sens_active, bounds=bounds_vars, method='highs')
print("Ефект збільшення активного ресурсу 1 на 15%:")
print(f"Новий виробничий план: x1 = {res_active_mod.x[0]:.2f}, x2 = {res_active_mod.x[1]:.2f}")
print(f"Новий прибуток: {-res_active_mod.fun:.2f} грн (приріст: {-res_active_mod.fun - (-opt_res.fun):.2f} грн)")

# Збільшення запасу неактивного ресурсу 3 на 50%
limits_sens_inactive = b_limits.copy()
limits_sens_inactive[2] *= 1.5

res_inactive_mod = linprog(c_obj, A_ub=A_constraints, b_ub=limits_sens_inactive, bounds=bounds_vars, method='highs')
print("\nЕфект збільшення неактивного ресурсу 3 на 50%:")
print(f"Виробничий план: x1 = {res_inactive_mod.x[0]:.2f}, x2 = {res_inactive_mod.x[1]:.2f}")
print(f"Прибуток: {-res_inactive_mod.fun:.2f} грн (зміна прибутку: {-res_inactive_mod.fun - (-opt_res.fun):.2f} грн)")


Ефект збільшення активного ресурсу 1 на 15%:
Новий виробничий план: x1 = 19.07, x2 = 67.79
Новий прибуток: 3760.36 грн (приріст: 292.50 грн)

Ефект збільшення неактивного ресурсу 3 на 50%:
Виробничий план: x1 = 23.57, x2 = 54.29
Прибуток: 3467.86 грн (зміна прибутку: 0.00 грн)


### Економічні висновки та аналіз чутливості
1. **Дефіцитні ресурси:** Ресурси 1 та 2 вичерпуються повністю (відповідні обмеження є активними). Саме вони утворюють "вузькі місця" виробничого процесу і лімітують подальше зростання прибутку.
2. **Надлишкові ресурси:** Ресурс 3 має вільний залишок, тому будь-яке нарощування його запасів не приносить жодного додаткового економічного ефекту.
3. **Управлінське рішення:** Додаткові інвестиції слід спрямовувати виключно на закупівлю активних ресурсів 1 та 2, оскільки їхнє розширення на 15% забезпечує відчутний прямий приріст прибутку підприємства.
